# Experiment 3: Full fine-tuning (Optimizer Mismatch)

No LoRA. All weights unfrozen. Directly tests the optimizer mismatch hypothesis.
If Muon hurts performance here but not in Experiment 1, LoRA protects against mismatch.

## Cell 1: Install

In [ ]:
!pip install -q transformers peft datasets accelerate trl nbformat
!pip install --upgrade Pillow

## Cell 2: Load Muon

In [ ]:
%run /home/ubuntu/thesis-storage-1/muon.ipynb

## Cell 3: Imports

In [ ]:
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
from transformers import AutoModelForCausalLM, AutoTokenizer
from datasets import load_dataset
from torch.utils.data import DataLoader
import copy
import os

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('VRAM:', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), 'GB')

## Cell 4: Config

In [ ]:
MODEL_NAME = 'microsoft/Phi-4-mini-instruct'

MAX_STEPS   = 7989   # 3 epochs
BATCH_SIZE  = 4      
MAX_SEQ_LEN = 512
GRAD_ACCUM  = 16     # keep effective batch size = 64

MUON_LR           = 0.002 # Replace it with a suboptimal LR. We replaced this with 2e-5 s mentioned in the moonlight paper but did not use the cosine decay func.
MUON_MOMENTUM     = 0.95
MUON_WD           = 0.1
MUON_UPDATE_SCALE = 0.3

ADAMW_LR = 2e-5 #Again Moonlight reccomendation
ADAMW_WD = 0.1

SVD_TRACK_EVERY = 1000

SAVE_DIR = '/home/ubuntu/thesis-storage-2/experiment_3_qwen_results' #Replace with ur file path
os.makedirs(SAVE_DIR, exist_ok=True)

print('Config set.')

## Cell 5: Load Tokenizer

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=False)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

## Cell 6: Load Dataset

In [ ]:
dataset = load_dataset('zwhe99/commonsense_170k', split='train')
print('Dataset size:', len(dataset))

def tokenize(example):
    text = f"### Instruction:\n{example['instruction']}\n### Response:\n{example['output']}"
    tokens = tokenizer(
        text, truncation=True, max_length=MAX_SEQ_LEN,
        padding='max_length', return_tensors='pt',
    )
    tokens['labels'] = tokens['input_ids'].clone()
    return {k: v.squeeze(0) for k, v in tokens.items()}

tokenized = dataset.map(tokenize, remove_columns=dataset.column_names)
tokenized.set_format('torch')
print('Tokenized. Batches per epoch:', len(tokenized) // BATCH_SIZE)

## Cell 8: Train Function

In [ ]:
def train(model, dataloader, optimizer_type, max_steps=MAX_STEPS, grad_accum=GRAD_ACCUM):
    print(f'\n===== Training with {optimizer_type.upper()} =====')

    if optimizer_type == 'muon':
        muon_params, adamw_params = get_muon_and_adamw_params(model)
        optimizer = [Muon(muon_params, lr=MUON_LR, momentum=MUON_MOMENTUM,
                         weight_decay=MUON_WD, update_scale=MUON_UPDATE_SCALE)]
        if len(adamw_params) > 0:
            optimizer.append(torch.optim.AdamW(adamw_params, lr=ADAMW_LR, weight_decay=ADAMW_WD))
    else:
        optimizer = [torch.optim.AdamW(model.parameters(), lr=ADAMW_LR, weight_decay=ADAMW_WD)]

    loss_history = []
    sv_history   = {}
    model.train()
    step = 0
    data_iter = iter(dataloader)

    while step < max_steps:
        try:
            batch = next(data_iter)
        except StopIteration:
            data_iter = iter(dataloader)
            batch = next(data_iter)

        input_ids      = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels         = batch['labels'].to(device)

        outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
        loss = outputs.loss / grad_accum
        loss.backward()

        if (step + 1) % grad_accum == 0:
            for opt in optimizer:
                opt.step()
                opt.zero_grad()

        loss_val = loss.item() * grad_accum
        loss_history.append((step, loss_val))

        if step % SVD_TRACK_EVERY == 0:
            sv_history[step] = track_svd_full(model, step)
            print(f'Step {step:4d} | Loss: {loss_val:.4f} | SVD tracked.')

        step += 1

    print(f'Final loss: {loss_val:.4f}')
    return loss_history, sv_history

## Cell 9: Run AdamW

In [ ]:
# Load fresh model
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.bfloat16,
    device_map='auto',
    trust_remote_code=False,
)
print('All parameters:', sum(p.numel() for p in model.parameters()) / 1e9, 'B')
print('All weights are trainable — no LoRA')

dataloader = DataLoader(tokenized, batch_size=BATCH_SIZE, shuffle=True)

# Save initial weights
initial_state = copy.deepcopy(model.state_dict())

adamw_losses, adamw_svd = train(model, dataloader, 'adamw')

torch.save({'losses': adamw_losses, 'svd': adamw_svd},
           os.path.join(SAVE_DIR, 'adamw_results.pt'))

## Cell 10: Run Muon

In [ ]:
# Reset to initial weights
model.load_state_dict(initial_state)
print('Model reset to initial weights.')

dataloader = DataLoader(tokenized, batch_size=BATCH_SIZE, shuffle=True)

muon_losses, muon_svd = train(model, dataloader, 'muon')

torch.save({'losses': muon_losses, 'svd': muon_svd},
           os.path.join(SAVE_DIR, 'muon_results.pt'))

## Cell 11: Plot Loss Curves

In [ ]:
adamw_steps = [x[0] for x in adamw_losses]
adamw_vals  = [x[1] for x in adamw_losses]
muon_steps  = [x[0] for x in muon_losses]
muon_vals   = [x[1] for x in muon_losses]
plt.figure(figsize=(10, 5))
plt.plot(adamw_steps, adamw_vals, label='AdamW', color='blue', alpha=0.8)
plt.plot(muon_steps,  muon_vals,  label='Muon',  color='orange', alpha=0.8)
plt.xlabel('Training Step')
plt.ylabel('Loss')
plt.title('Full Fine-Tuning Loss: Muon vs AdamW (Phi-4-mini, No LoRA)')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'loss_curves.png'), dpi=150)
plt.show()